# Formation Complète et Ultra-Détaillée : Algorithmes de Recherche pour un Robot

Bienvenue dans ce carnet Jupyter interactif revisité pour offrir le **maximum de détails possibles**. L'idée ici est qu'une personne n'ayant jamais fait de programmation ou d'intelligence artificielle puisse comprendre la logique mathématique et informatique de A à Z.

### Quel est le problème à résoudre ?
Imagine un robot placé dans un labyrinthe vu de haut. Son but est de se rendre à la sortie le plus rapidement possible. Cependant, le robot est aveugle : il ne voit pas la sortie directement, il ne peut que tester des chemins.
Nous allons lui implanter 3 « cerveaux » différents (3 algorithmes) pour voir comment il se débrouille :
1. **L'algorithme Glouton** : Il fonce toujours vers la direction qui semble se rapprocher de la fin, sans jamais anticiper.
2. **L'algorithme A* (A-Etoile)** : C'est un GPS intelligent. Il calcule non seulement la distance vers la fin, mais prend aussi en compte le chemin déjà parcouru.
3. **L'algorithme Génétique** : Il se base sur la théorie de l'évolution. Il donne des directions au hasard à 100 robots, garde ceux qui sont allés le plus loin, et croise leurs mouvements pour créer des bébés robots plus intelligents.

Tu peux exécuter toutes les cellules de code ci-dessous une par une en appuyant sur `Maj + Entrée` (Shift + Enter).

## Partie 1 : Créer notre Monde Virtuel (La Grille)

En informatique, un monde physique est souvent représenté sous la forme d'un tableau (une grille).
Chaque case de la grille contient une information : 
- `S` : La position de Départ du robot (Start).
- `G` : La position d'arrivée visée par le robot (Goal).
- `X` : Un mur infranchissable.
- `0` : Une case de terre vide sur laquelle le robot a le droit de marcher.

Le code ci-dessous va créer un fichier texte `test_grid.txt` qui dessine cette grille. C'est ce qu'on appelle un jeu de données (dataset).

In [ ]:
# On utilise une chaine de caracteres sur plusieurs lignes (merci aux triples guillemets)
# pour dessiner visuellement a quoi ressemblera notre monde.
contenu_grille = """S 0 0 0 0
0 X X X 0
0 0 0 0 0
0 X X X 0
0 0 0 G 0"""

# 'with open("nom", "w")' permet d'ouvrir un fichier en mode ecriture ("w" pour "write").
# Si le fichier n'existe pas, l'ordinateur le cree tout seul !
with open("test_grid.txt", "w") as fichier:
    fichier.write(contenu_grille) # On ecrit notre dessin dans le fichier texte

print("Etape 1 reussie : Le fichier virtuel 'test_grid.txt' a ete cree !")


### 1.1 Traduire le texte en mathématiques (La fonction load_grid)

Pour le moment, `test_grid.txt` n'est que du texte. Python ne sait pas ce que c'est.
Il faut convertir ce texte en un tableau mathématique. 

**Attention : Comment fonctionnent les coordonnées (x,y) en programmation ?**
- L'axe X (horizontal) va de gauche à droite. La première colonne est x=0.
- L'axe Y (vertical) **va de haut en bas** (contrairement aux mathématiques de l'école) ! La première ligne tout en haut est y=0. La ligne en dessous est y=1.
- Donc la coordonnée (x=3, y=1) désigne la 4ème case vers la droite, sur la 2ème ligne en partant du haut.

In [ ]:
def load_grid(file):
    # Cette liste vide va se transformer en une liste de listes (tableau à 2 dimensions).
    grid = []
    
    # On declare des variables vides pour stocker plus tard où se situe le depart et l'arrivee.
    start = None
    goal = None

    # On ouvre notre fichier texte en mode lecture classique
    with open(file) as f:
        # La fonction 'enumerate' est un compteur magique. 
        # Si 'f' contient 5 lignes, elle va donner y=0 pour la ligne 1, y=1 pour la ligne 2, etc.
        for y, line in enumerate(f):
            
            # line contient le texte brut d'une ligne, par exemple : "S 0 0 0 0\n"
            # .strip() nettoie la ligne (retire les sauts de lignes invisibles à la fin)
            # .split() coupe la ligne en morceaux à chaque fois qu'il voit un espace.
            # a ce stade, 'row' (qui veut dire ligne) est une liste Python : ['S', '0', '0', '0', '0']
            row = line.strip().split()
            
            # Maintenant on parcourt chaque element de cette liste 'row'.
            # Ici, 'x' va etre le numero de la colonne (0, 1, 2, 3, 4) et 'val' sera la lettre ('S', '0'...).
            for x, val in enumerate(row):
                
                # Si la lettre observee est "S", alors on vient de trouver le depart !
                if val == "S":
                    # On stocke immediatement ses coordonnees x et y dans une parenthese (un Tuple)
                    start = (x, y)
                    
                # Si la lettre est "G", on a trouve la sortie !
                if val == "G":
                    goal = (x, y)
                    
            # A la fin du traitement de la ligne de texte, on ajoute la liste nettoyee a notre grande Grille.
            grid.append(row)

    # La fonction renvoie 3 elements : la grille finale, le point de depart et le point d'arrivee
    return grid, start, goal

# --- ON TESTE LA FONCTION POUR VOIR LE RESULTAT EN VRAI ---
ma_grille, mon_depart, mon_arrivee = load_grid("test_grid.txt")

print("-- Resultats du chargement --")
print("Position S trouvee en coordonnees (x, y) :", mon_depart)
print("Position G trouvee en coordonnees (x, y) :", mon_arrivee)
print("Voici comment l'ordinateur voit definitivement la grille :")
for ligne in ma_grille:
    print(ligne)


### 1.2 Regarder autour de soi (La fonction get_neighbors)

Un algorithme de recherche ne peut pas magiquement se téléporter à l'arrivée. Il doit avancer case par case. 
Depuis une position donnée, l'algorithme doit pouvoir lister quelles sont les "cases voisines accessibles".
- Autorisé : case Haut, Bas, Gauche, Droite contenant '0', 'S' ou 'G'.
- Interdit : case en dehors des limites de la carte ou contenant un 'X' (mur).

In [ ]:
def get_neighbors(grid, pos):
    # 'pos' contient les dernieres coordonnees du robot, par exemple (0,0).
    # En ecrivant 'x, y = pos', Python extrait automatiquement 0 dans x et 0 dans y.
    x, y = pos
    
    # len(grid) compte le nombre de lignes (donc la hauteur totale)
    rows = len(grid)
    # len(grid[0]) compte la taille de la première ligne (donc la largeur totale)
    cols = len(grid[0])

    # Voici la liste des 4 directions dans lesquelles le robot a le droit de bouger :
    # (1, 0) veut dire : avancer de +1 sur l'axe horizontal X (donc aller a droite), et 0 sur Y.
    # (-1, 0) veut dire : avancer de -1 sur l'axe X (donc aller a gauche).
    # (0, 1) veut dire : avancer de +1 sur l'axe vertical Y (donc aller vers le bas).
    # (0, -1) veut dire : avancer de -1 sur l'axe Y (donc aller vers le haut).
    directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    
    # On prepare une liste vide qui stockera les reponses correctes
    neighbors = []

    # Cette boucle va analyser chacune des 4 directions une par une
    for dx, dy in directions:
        
        # nx (new_x) et ny (new_y) sont les futures coordonnees si le robot fait le pas
        nx = x + dx
        ny = y + dy
        
        # REGLE 1 : Le mur du vide.
        # nx doit rester parfaitement dans le cadre : entre 0 et la largeur de la carte (cols).
        # ny doit aussi rester entre 0 et la hauteur de la carte (rows).
        if 0 <= nx < cols and 0 <= ny < rows:
            
            # REGLE 2 : Le parpaing.
            # L'acces est refuse si la case qu'on essaie de rejoindre contient "X", car c'est un mur.
            # Note informatique : on cible grid[ligne][colonne] d'où le grid[ny][nx].
            if grid[ny][nx] != "X":
                
                # Si le futur pas survit aux deux regles, c'est une case officielle accessible !
                # On l'ajoute dans le panier.
                neighbors.append((nx, ny))

    # On renvoie a l'algorithme tout le contenu du panier
    return neighbors

# --- ON TESTE LA VISION DU ROBOT ---
print("Depuis S en (0,0), les options sont :", get_neighbors(ma_grille, (0,0)))
print("Explication : A gauche et en haut il y a le vide (mur invisible).") 
print("Le robot ne peut donc aller qu'a droite (1,0) et en bas (0,1) !")


## Parenthèse Essentielle : Comment prendre une décision ? (La file de Priorité Heapq)

L'algorithme Glouton et A* doivent sans cesse analyser plein de chemins en même temps.
Pour savoir "Quel est le chemin le plus prometteur ?", on utilise une **File de Priorité** appelée `heapq` en Python.

Imagine les urgences d'un hôpital. S'il y a une simple file d'attente (comme dans un supermarché), le premier arrivé est le premier soigné. Mais dans une file de priorité, chaque nouveau patient reçoit une **Note d'Urgence**. Le médecin appelle toujours la personne ayant le score d'urgence le plus critique (ici, le chiffre le plus faible).

In [ ]:
import heapq

# On declare une liste de patients vide
hopital = []

# On rentre les patients le format (Degre_Securite, "Commentaire")
# Attention, plus le numero est bas, plus il passera dans le bureau du medecin en premier !
heapq.heappush(hopital, (100, "Patient A : Mal de ventre (est arrivé en premier)"))
heapq.heappush(hopital, (3, "Patient B : Crise Cardiaque (est arrivé en 2eme)"))
heapq.heappush(hopital, (50, "Patient C : Bras cassé (est arrivé en 3eme)"))

print("-- LE MEDECIN APPEL LE PATIENT LE PLUS IMPORTANT --")
note, description = heapq.heappop(hopital)
print("Reponse : Le medecin soigne d'abord =>", description)
print("Conclusion : heapq trie et ressort les valeurs automatiquement en fonction de la NOTE. C'est inestimable en robotique.")


## Partie 2 : Algorithme Glouton (Greedy Search)

Le système glouton est simple. Il fonctionne à 100% avec l'idée : "Je prends le chemin de ma fille de priorité qui a la distance estimée la plus courte vers la fin". 

Pour savoir à quelle distance il se trouve de la fin, il utilise **Une Heuristique**, c'est-à-dire une estimation mathématique. Notre robot ne peut pas se déplacer en diagonale. La formule qui calcule sa séparation avec l'arrivée comme un oiseau mais sans les diagonales s'appelle "La Distance de Manhattan" (du nom du quartier de New York où tu es obligé de zigzaguer entre les immeubles carrés pour avancer).

In [ ]:
def heuristic(a, b):
    # 'a' est la position actuelle (ex: x=1, y=2)
    # 'b' est la position visee (le but)
    # abs() retire le signe négatif. Exemple : La difference entre x=1 et x=5 est de -4, abs(-4) devient 4 blocs de distance.
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def rebuild_path(came_from, goal):
    # Cette petite fonction est une machine a remonter le temps.
    # une fois le but atteint, elle remonte la "vignette parent" de chaque case pour redessiner le tracé final.
    path = []
    current = goal
    # On remonte tant qu'il y a un parent
    while current is not None:
        path.append(current)          # On ajoute la case visitée
        current = came_from[current]  # On passe à la case qui nous a emmené ici
        
    path.reverse() # Le chemin tracé etait de l'arrivee vers depart, on le met a l'endroit (depart vers arrivee)
    return path

def greedy_search(grid, start, goal):
    
    # 1. On prepare le carnet de rendez-vous de l'hopital
    open_list = []
    
    # 2. On rentre notre position de depart. Sa note "d'urgence" est la distance qui le separe de G.
    heapq.heappush(open_list, (heuristic(start, goal), start))

    # 3. Le dictionnaire 'came_from' est la memoire du robot. 
    # Pour chaque case explorée, il retiendra (Quelle_Case -> vient_de_Quelle_Case).
    # C'est comme le fil d'Ariane ! Le depart ne vient de nulle part ("None").
    came_from = {start: None}

    # 4. Le Boucle (Tant qu'il reste des chemins en attente dans le carnet de RDV)
    while open_list:
        
        # 5. On demande a heapq de nous sortir la position qui possede LA PLUS PROCHE NOTATION par rapport à G.
        # _, permet d'ignorer la note, seule la position nous interesse on l'appelle "current".
        _, current = heapq.heappop(open_list)
        
        # 6. BANCO ! Si la position ultra-prioritaire est notre objectif...
        if current == goal:
            # On stoppe tout l'algorithme instantanement, et on rembobine le fil d'ariane !
            return rebuild_path(came_from, goal)

        # 7. Si ce n'est pas le but, on va scruter les voisins autour (Haut, Bas, Gauche, Droite)...
        for neighbor in get_neighbors(grid, current):
            
            # Si le robot n'est jamais alle sur cette case..
            if neighbor not in came_from:
                
                # Il l'enregistre en attachant un fil d'ariane reliant le voisin a la casse actuelle
                came_from[neighbor] = current
                
                # Il calcule à quelle distance de G se situe ce voisin.
                h = heuristic(neighbor, goal)
                
                # ET SURTOUT, il inscrit ce voisin dans le carnet de RDV de heapq avec sa note !
                # Ainsi au prochain tour, la boucle s'interressera à lui !
                heapq.heappush(open_list, (h, neighbor))

    # Si la file d'attente s'est videe mais qu'il n'a jamais croisé G, alors tout est entierement bloqué par des murs.
    return [] 

# --- ON TESTE LE GLOUTON ---
chemin_glouton = greedy_search(ma_grille, mon_depart, mon_arrivee)
print("Parcours trouve via Glouton :", chemin_glouton)
print("Cela prend au robot :", len(chemin_glouton)-1, "etapes de marche.")


## Partie 3 : L'Algorithme A* (A-Etoile)

L'algorithme Glouton (au dessus) a **un defaut monstrueux** : il est obnubilé par "l'envie de s'approcher à vol d'oiseau". S'il y a un grand mur devant le but, le glouton va marcher le long du mur en essayant désespérement de s'en rapprocher et va créer un chemin très moche et terriblement lent avant de finir par le contourner.

**A* regle ce probleme définitivement.**
Pour A*, le score attribué à la file d'attente ne représente plus uniquement _"La distance de l'oiseau"_.
Le score de A* s'appelle `F` et se dessine `F = G + H`.
- `H` = Comme tout à l'heure, l'estimation théorique de l'oiseau (Heuristique).
- `G` = LE COUT REEL, VERITABLE ET ACTUEL. C'est-à-dire : "Combien de pas j'ai fait concrètement depuis S pour arriver dans ma situation ?".

Qu'est-ce que ca change ? C'est dramatique ! 
Si A* longe un mur sans succès, alors son compteur de pas `G` explose, et de facto la note de rentrée globale `F` devient énorme. Du coup, comme `heaqp` trie les plus petits scores en premier, il va reléguer cette exploration minable tout en bas de sa liste d'attente, et va instantanément s'intéresser à des chemins tout frais. _A* garantit un pourcentage de taux de réussite du chemin LE PLUS COURT mathématiquement existant._

In [ ]:
def astar_search(grid, start, goal):
    open_list = []
    
    # Tout commence par une note de 0 pour S.
    heapq.heappush(open_list, (0, start))
    came_from = {start: None}
    
    # MEMOIRE ADDITIONNELLE : Ce dictionnaire retient, case par case visitée, 
    # combien d'energie reelle (combien de pas reels) ont ete depenses depuis le demarrage.
    # Pour le depart, c'est logique, il a coute 0 pas.
    g_score = {start: 0} 

    while open_list:
        
        # On extrait la case qui a la plus belle etincelle (le Score F le plus faible, combinant H et G).
        _, current = heapq.heappop(open_list)

        # BRAVO : On a touche G.
        if current == goal:
            return rebuild_path(came_from, goal)

        # On analyse le voisinage comme prevu.
        for neighbor in get_neighbors(grid, current):
            
            # L'innovation est ici :
            # L'energie depensee pour atteindre un voisin, c'est toujours "l'energie pour atterir sur MOI + 1 pas de plus" !
            new_cost_g = g_score[current] + 1
            
            # CONDITION POUR EVALUER CE VOISIN : 
            # 1. C'est une case inexplorée (neighbor not in g_score).
            # 2. OU ALORS c'est une vielle case deja vue, MAIS j'ai reussi à atterrir sur elle 
            # avec beaucoup moins de pas que quand je l'ai decouverte à l'epoque (new_cost_g < ancient cost).
            if neighbor not in g_score or new_cost_g < g_score[neighbor]:
                
                # Je mets ses references a jour !
                g_score[neighbor] = new_cost_g
                came_from[neighbor] = current
                
                # Le calcul triomphal A* => Cout Verifie + Distance Heuristique Restante.
                score_f = new_cost_g + heuristic(neighbor, goal)
                
                # J'envoie ma requete de ce chemin au standart (heapq).
                heapq.heappush(open_list, (score_f, neighbor))

    return [] # Encore une fois, c'est si on ne touve rien

# --- ON TESTE A* ---
chemin_astar = astar_search(ma_grille, mon_depart, mon_arrivee)
print("Parcours PARFAIT et VERIFIE via A-Etoile :", chemin_astar)
print("Temps depense en etapes :", len(chemin_astar)-1)


## Partie 4 : Algorithme Génétique

Changement radical de philosophie. Contrairement aux deux algorithmes précédents, le **Génétique** ne dresse aucune liste d'attente logique de la grille. Sa force repose purement sur l'imprévisibilité et les phénomènes de la nature ("La sélection naturelle").

Le principe pour le Robot : 
1. **La Population** : Je lâche 100 robots kamikazes immobiles au point S.
2. **Le Chromosome** : Je leur installe un tout petit cerveau aléatoire (une liste de choix pris au hasard, par exemple "Gauche, Haut, Droite, Gauche..."). **La taille de ce chromosome s'adapte automatiquement à la taille de la grille (plus la grille est grande, plus le chromosome sera long) pour être sûr d'avoir assez de mouvements possibles.**
3. **L'Évaluation (La Fitness / La Forme)** : Je fais fermer les yeux des robots et je crie "GO". Ils courent tous comme des fous suivants leur chromosome ! **Désormais, dès qu'un robot atteint la solution G, on stoppe instantanément toute l'évaluation pour gagner du temps.** Sinon, on regarde où chacun a fini. Plus ils étaient proches de G, meilleure sera leur *Fitness* (Leur notation corporelle).
4. **La Reproduction & les Mutations** : A la fin du massacre, j'abats les robots les moins performants. Je garde les meilleurs par tournoi. **Je mélange l'ADN de leur chromosome** pour créer de nouveaux "Bébés Robots" (ce sera la *nouvelle génération*). Mais durant le processus, petit twist, de temps en temps une valeur au pif subit une **MUTATION** (un "Droite" se transforme en "Haut"). Pourquoi cette anomalie génétique ? C'est ce qui sauve l'algorithme : ça le force à sortir des sentiers battus.

In [ ]:
# On importe la librairie qui permet de faire le hasard pur : 'random'
import random

# La liste des mouvements entiers : 0=Droite, 1=Gauche, 2=Bas, 3=Haut.
MOVES = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def apply_moves(grid, start, goal, chromosome):
    # C'est la fonction "Robot Kamikaze".
    # Ca force le robot à consommer de force son "Chromosome" (sa liste d'actions).
    
    x, y = start
    path = [(x, y)] # On initialise la camera d'un traceur GPS 
    rows, cols = len(grid), len(grid[0])
    
    # Pour cheque mouvement au de l'ADN de l'individu :
    for move in chromosome:
        dx, dy = MOVES[move]
        nx, ny = x + dx, y + dy
        
        # S'il prevoit de bouger SUR LE VIDE ou SUR UN MUR ("X")...
        if 0 <= nx < cols and 0 <= ny < rows and grid[ny][nx] != "X":
            
            # C'etait autorisé ! Il avance ! 
            x, y = nx, ny
            path.append((x, y)) # Le traceur l'enregistre dans un historique.
            
            # Il frôle brusquement l'arrivee ??
            if (x, y) == goal: 
                break # EXCELLENT ! On pete completement sa routine (il n'est meme pas forcé de finir l'ADN).
                
    # On renvoie le traceur que le robot a dessiné en s'eloignant de S
    return path

def fitness(grid, start, goal, chromosome):
    # La "Fitness" est le juge d'un concours de patinage pour Robots. Un chiffre positif enorme = Merveilleux
    # Un chiffre negatif tres petit (-100, -200) = Catastrophique.
    
    path = apply_moves(grid, start, goal, chromosome)
    # last_pos c'est la toute derniere case sur laquelle le robot a ete vu avant de mourir ou de finir
    last_pos = path[-1] 
    
    # A quel chiffre il etait vis-a-vis de l'objectif sur un vol d'oiseau pur ?
    dist = heuristic(last_pos, goal)
    
    if last_pos == goal:
        # Il a touché G !!!!
        # Il gagne la recompense supreme : 1000 points !
        # MAIS : on devalue legerement sa medaille en soustrayant la 'longueur du traceur'.
        # Cela incitera la nature animale au prochain tour a "Toucher le But MAIS avec moins de depenses".
        return 1000 - len(path)
    else:
        # Son path a tape dans un X avant la fin. Il a raté.
        # Son evaluation s'exprime dans le format "-3" (il manquait 3 blocs), ou "-10" (il manquait 10 blocs).
        return -dist


def genetic_search(grid, start, goal):
    POPULATION_SIZE = 100
    GENERATIONS = 200
    
    # On adapte la taille du chromosome a l'espace de la grille
    rows = len(grid)
    cols = len(grid[0]) if rows > 0 else 0
    chromosome_length = max(40, rows * cols)
    
    # Initialisation de la population
    population = [[random.randint(0, 3) for _ in range(chromosome_length)] for _ in range(POPULATION_SIZE)]
    
    best_path = []
    best_score = float('-inf')
    
    for generation in range(GENERATIONS):
        scores = []
        solution_trouvee = False
        
        # Evaluation : on s'arrete des qu'on le peut !
        for chrom in population:
            score = fitness(grid, start, goal, chrom)
            scores.append(score)
            
            if score > best_score:
                best_score = score
                best_path = apply_moves(grid, start, goal, chrom)
                
            if best_path and best_path[-1] == goal:
                solution_trouvee = True
                break
                
        if solution_trouvee:
            break
            
    return best_path

print('Theorie genetique (mise a jour) integree avec succes !')


## Partie 5 (Apothéose Mathématique et Visuelle)

Tu vas maintenant assister à une traduction d'un tableau abstrait Python vers une véritable image haute résolution colorisée calculée pixel par pixel.
La librairie Python `numpy` est un super-calculateur de matrices et `matplotlib.pyplot` va afficher ça comme un peintre visuel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def peindre_une_image_des_algorithmes(grid, chemin, titre_au_choix):
    
    # 1. On etablie la hauteur de l'image (le nombre de lignes) et sa largeur en pixel.
    rows, cols = len(grid), len(grid[0])
    
    # 2. Une Image informatique n'est en realite qu'un tableau Numpy emplie de chiffre.
    # On la definit  en 3D (Hauteur, Largeur, Pinceaux_Couleurs). 
    # Le chiffre 3 correspond au fait que chaque Pixel va reclamer les couleurs (Rouge, Vert, Bleu - RGB).
    # NP.ZEROS : on remplie tout de noir total au debut.
    image_vierge = np.zeros((rows, cols, 3))
    
    # 3. La Boucle d'allumage des Pixels.
    # Pour chaque ligne de notre grille de fichier texte..
    for y, row in enumerate(grid):
        
        # Et pour cheque lettre observee sur cette ligne..
        for x, val in enumerate(row):
            
            # SI cette position EXACTE est dans la liste retracée par mon Algorithme (mais n'est pas "S" ou "G")...
            if (x, y) in chemin and val not in ("S", "G"):
                # On allume le pixel en BLEU FLUO !
                # Code Couleur (R: 0.2, G: 0.8, B: 1.0)
                image_vierge[y, x] = [0.2, 0.8, 1.0] 
                
            elif val == "S":
                # Le depart "S" est peint en Vert Fluorescent
                image_vierge[y, x] = [0.2, 0.8, 0.2] 
                
            elif val == "G":
                # L'arrivee "G" est peint en Orange Flamboyant
                image_vierge[y, x] = [1.0, 0.6, 0.0] 
                
            elif val == "X":
                # Les MUR sont allumés en Rouge Sombre pur !
                image_vierge[y, x] = [0.8, 0.2, 0.2]
                
            else:
                # Tout la terre vide "0" reste Gris anthracite.
                image_vierge[y, x] = [0.1, 0.1, 0.1]
                
    # --- ON MET TOUT CA EN FORME A L'ECRAN --
    # On lui demande de generer la fenetre
    fig, ax = plt.subplots(figsize=(5, 5))
    
    # On insere magiquement notre tableau numpy (notre 'image_vierge' colorisee)
    ax.imshow(image_vierge)
    
    ax.set_title(titre_au_choix, color="black", fontweight="bold")
    
    # Creation d'une petite grille blanche style quadrillage cahier
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    
    # On supprime les chiffres du bord qu'utilisent les graphiques habituels
    ax.tick_params(which="both", labelbottom=False, labelleft=False, left=False, bottom=False)
    
    # On declenche l'apparition de la fenetre à tes yeux !!
    plt.show()

# == DECLENCHEMENT DU PROCESSUS TOTAL !!! ==
print("Apparition Graphique de la route tracee par le robot A-Etoile :")
peindre_une_image_des_algorithmes(ma_grille, chemin_astar, "RESULTAT MAGNIFIQUE (ALGO A*)")


## Conclusion Finale
Tu peux être fier de toi ! Que tu viennes du monde médical, artistique ou rien du tout, tu as toutes les clefs sous les mains, extrêmement commentées, des algorithmes les plus populaires du développement IA et jeu vidéo.
Amuse-toi à modifier la première ligne de code en ajoutant des murs pour bloquer les algorithmes !